In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [2]:
#files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
files = np.array(sorted(glob.glob('/home/ulyanov/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))

dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,4,8),
#                     dates < datetime(2026,5,3)], axis=0))[0] # CR2310

#t = np.where(np.all([dates > datetime(2026,7,15),
#                     dates < datetime(2026,9,6)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,20),
                     dates < datetime(2026,10,6)], axis=0))[0] # CR2315


files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/4science/blos/phi-fdt-blos_20260820T002003_V202610201927C_0648200021.fits'),
 np.str_('/home/ulyanov/data/4science/blos/phi-fdt-blos_20261005T093003_V202610211201C_0650050504.fits'),
 134)

In [ ]:
data, view = make_map(files[:], kind='synoptic', correct_dr=True, correct_bias=False, car_rot=2, a=360 / 15, b=0, c=0)

In [ ]:
fig, ax = show_data(data, view, cmap='seismic', vmin=-20, vmax=20)

In [ ]:
Q = []
T_range = np.arange(20, 41)
for T in T_range:
    print(T)

    data, view = make_map(files[:], kind='synoptic', correct_dr=True, car_rot=2, a=360 / T, b=0, c=0)
    Q += [data]

    np.savez('Q.npz', Q=np.array(Q))

Q = np.array(Q)

In [ ]:
Q = np.load('Q.npz')['Q']

In [ ]:
nx = Q.shape[-2]
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
np.savez('diff_rot.npz', data=Q, T=T_range, lat=lat)

In [3]:
from utils import rebin

s = np.load('diff_rot.npz')
#s = np.load('/home/ulyanov/data/4science/CR2315_diff_rot.npz')
Q = s['data']
T = s['T']
lat = s['lat']

#Q = rebin(Q, 2, axis=1)
#lat = lat[::2] + 0.5

In [4]:
Q.shape, lat.shape

((26, 720, 1800), (720,))

In [7]:
A, B, C = 14.712, -2.396, -1.787

q = np.nanstd(Q[5:], axis=-1)
p = np.nanargmax(np.nan_to_num(q), axis=0)

y = np.sin(lat * np.pi / 180)

plt.figure(figsize=(10,8))
plt.plot(y, 360 / T[5:][p], lw=2, label='FDT, CR2315')
plt.plot(y, A + B * y ** 2 + C * y ** 4, '--', lw=2, label='Snodgrass, 1990')

plt.xlabel('Latitude, degrees')
plt.ylabel('Rotation rate, deg/day')

plt.legend()
plt.grid(True)
plt.xlim(0,1)
plt.ylim(8,16)
plt.tight_layout()

In [12]:
from scipy.ndimage import gaussian_filter

q = np.nanstd(Q[5:], axis=-1)
q = q / np.max(q, axis=0, keepdims=True)
p = np.nanargmax(np.nan_to_num(q), axis=0)


fig, ax = plt.subplots(figsize=(10,8))
ax.imshow(q, vmin=0.7, vmax=1,
          origin='lower', aspect='auto', cmap='jet', interpolation='none', extent=(-1, 1, T[5] - 0.5, T[-1] + 0.5))
#ax.plot(y, 20 + p, '--', lw=2, color='white')
#ax.plot(y, A + B * y ** 2 + C * y ** 4, '--', lw=2, label='Snodgrass, 1990')

ax.set_xticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180))
ax.set_xticklabels(np.arange(-90, 91, 15))
plt.tight_layout()

In [11]:
q = np.nanstd(Q[5:], axis=-1)

plt.figure(figsize=(10,8))
plt.plot(T[5:], q[:,300])
plt.tight_layout()